In [ ]:
# ============================================
# AP6 - QWEN VIA OLLAMA COM FINE-TUNING - VERSÃO CORRIGIDA
# ============================================

import requests
import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, classification_report
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import os
import sys
import re
import time
import subprocess
import threading
import tempfile

# ============================================
# 0. INSTALAÇÃO E CONFIGURAÇÃO DO OLLAMA
# ============================================

def setup_ollama():
    """Configura o Ollama no Colab ou ambiente local"""
    try:
        from google.colab import files
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    if IN_COLAB:
        print("="*60)
        print("INSTALANDO OLLAMA NO GOOGLE COLAB")
        print("="*60)

        print("\n1. Instalando dependências...")
        !apt-get update -qq
        !apt-get install -y zstd

        print("\n2. Instalando Ollama...")
        !curl -fsSL https://ollama.com/install.sh | sh

        os.environ['PATH'] += ":/usr/local/bin"

        def run_ollama_server():
            subprocess.run(["ollama", "serve"], capture_output=True)

        server_thread = threading.Thread(target=run_ollama_server, daemon=True)
        server_thread.start()

        print("\n3. Aguardando servidor Ollama iniciar...")
        time.sleep(15)

        max_attempts = 10
        for attempt in range(max_attempts):
            try:
                response = requests.get("http://localhost:11434/api/tags", timeout=5)
                if response.status_code == 200:
                    print("✅ Ollama server está rodando!")
                    break
            except:
                time.sleep(3)
                print(f"  Tentativa {attempt+1}/{max_attempts}...")

        print("\n4. Baixando modelo base...")
        !ollama pull qwen3:4b
        !ollama pull mxbai-embed-large

        print("\n✅ Ollama pronto para uso!")

    return IN_COLAB

IN_COLAB = setup_ollama()

# ============================================
# CONFIGURAÇÕES
# ============================================

OLLAMA_URL = "http://localhost:11434"
MODELO_BASE = "qwen3:4b"
MODELO_FINETUNED = "qwen3-stil-finetuned"  # Nome base sem :latest
MODELO_FINETUNED_COMPLETO = "qwen3-stil-finetuned:latest"  # Nome completo com :latest
MODELO_EMBEDDING = "mxbai-embed-large"

# Verificar qual modelo está disponível
def verificar_modelo_disponivel():
    """Verifica se o modelo fine-tunado está disponível"""
    try:
        response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10)
        if response.status_code == 200:
            modelos = response.json().get('models', [])
            for m in modelos:
                nome = m.get('name', '')
                # Verificar tanto com :latest quanto sem
                if nome == MODELO_FINETUNED_COMPLETO or nome == MODELO_FINETUNED:
                    print(f"✅ Modelo encontrado: {nome}")
                    return nome
            print(f"⚠️ Modelo {MODELO_FINETUNED} não encontrado.")
            print(f"   Modelos disponíveis: {[m.get('name') for m in modelos]}")
            return None
    except Exception as e:
        print(f"Erro ao verificar modelos: {e}")
    return None

# Verificar modelo
MODELO_DISPONIVEL = verificar_modelo_disponivel()
if MODELO_DISPONIVEL:
    MODELO_USAR = MODELO_DISPONIVEL
else:
    MODELO_USAR = MODELO_BASE
    print(f"Usando modelo base: {MODELO_USAR}")

print(f"\nModelo selecionado: {MODELO_USAR}")

# ============================================
# 1. CARREGAR CORPUS DO JSON
# ============================================

print("\n" + "="*60)
print("1. CARREGANDO CORPUS DO JSON")
print("="*60)

json_path = "stil2023_articles_limpo.json"

if not os.path.exists(json_path):
    arquivos_json = [f for f in os.listdir('.') if f.endswith('.json')]
    if arquivos_json:
        json_path = arquivos_json[0]
        print(f" Usando: {json_path}")
    elif IN_COLAB:
        from google.colab import files
        print(" Faça upload do arquivo JSON:")
        uploaded = files.upload()
        json_path = next(iter(uploaded.keys()))
    else:
        print(" Arquivo JSON não encontrado!")
        sys.exit(1)
else:
    print(f" Arquivo encontrado: {json_path}")

with open(json_path, encoding='utf-8') as f:
    articles = json.load(f)

print(f" Artigos carregados: {len(articles)}")

# ============================================
# 2. CRIAR DATASET DE TREINO (150 EXEMPLOS)
# ============================================

print("\n" + "="*60)
print("2. CRIANDO DATASET DE TREINO (150 EXEMPLOS)")
print("="*60)

# ==========================================
# ESTILO ACADÊMICO (50 exemplos)
# ==========================================

textos_academico = [
    "Observou-se que os resultados apresentam significância estatística.",
    "Foi verificado que os modelos baseados em transformer superam abordagens anteriores.",
    "Conclui-se que a metodologia empregada demonstra eficácia na tarefa proposta.",
    "Os dados foram coletados e analisados estatisticamente segundo protocolos estabelecidos.",
    "Realizou-se uma análise detalhada dos componentes principais do modelo.",
    "Verificou-se uma correlação significativa entre as variáveis analisadas.",
    "Pode-se observar que os resultados obtidos são consistentes com a literatura.",
    "Nota-se uma tendência clara de melhoria no desempenho dos classificadores.",
    "Foi demonstrado que a abordagem proposta é eficaz para o problema em questão.",
    "Conduziu-se um experimento controlado para avaliar o impacto das configurações.",
    "É importante ressaltar que os resultados devem ser interpretados com cautela.",
    "Considera-se que a amostra utilizada é representativa da população estudada.",
    "Entende-se que os achados contribuem significativamente para a área de conhecimento.",
    "Salienta-se a necessidade de replicação dos experimentos em diferentes contextos.",
    "Destaca-se a relevância dos resultados obtidos para aplicações práticas.",
    "Ressalta-se que as limitações do estudo não comprometem as conclusões principais.",
    "Argumenta-se que os modelos neurais apresentam vantagens sobre métodos tradicionais.",
    "Sugere-se que pesquisas futuras investiguem a generalização dos resultados.",
    "Infere-se dos dados que existe uma relação causal entre as variáveis estudadas.",
    "Depreende-se da análise que os resultados são robustos a diferentes configurações.",
    "A análise estatística foi realizada utilizando o software R versão 4.0.",
    "Os experimentos foram conduzidos em ambiente controlado com temperatura constante.",
    "A métrica de avaliação utilizada foi a acurácia balanceada devido ao desbalanceamento.",
    "O conjunto de dados foi dividido em treino (70%), validação (15%) e teste (15%).",
    "A significância estatística foi avaliada utilizando o teste t de Student (p<0,05).",
    "Os intervalos de confiança foram calculados utilizando o método bootstrap com 1000 replicações.",
    "A validação cruzada com 10 folds foi empregada para avaliar a estabilidade do modelo.",
    "O pré-processamento incluiu remoção de stopwords, stemming e normalização de caixa.",
    "A matriz de confusão revelou que o modelo apresenta alta precisão e recall.",
    "A curva ROC apresentou AUC de 0,95, indicando excelente poder discriminativo.",
    "Os resultados indicam que a hipótese nula foi rejeitada (p<0,001).",
    "A análise de variância mostrou diferenças significativas entre os grupos experimentais.",
    "O coeficiente de correlação de Pearson foi de 0,87 (p<0,01), indicando forte correlação.",
    "Os modelos baseados em BERT superaram significativamente as abordagens baseline (p<0,05).",
    "A acurácia média do modelo proposto foi de 94,5% (DP=1,2%) nos dados de teste.",
    "Os resultados sugerem que o método proposto é robusto a ruídos nos dados de entrada.",
    "A análise de sensibilidade mostrou que o modelo é estável para variações nos parâmetros.",
    "Os experimentos de ablação confirmaram a importância de cada componente do sistema.",
    "A validação externa em um corpus independente confirmou a generalização dos resultados.",
    "Conclui-se que a abordagem proposta é promissora para aplicações em PLN.",
    "Foi observada uma melhoria consistente de 15% em relação ao estado da arte.",
    "Realizou-se uma busca sistemática na literatura para identificar trabalhos relacionados.",
    "A métrica F1 foi escolhida como medida principal devido ao desbalanceamento das classes.",
    "O modelo foi treinado por 50 épocas, com early stopping baseado na perda de validação.",
    "A curva de aprendizado mostrou convergência após aproximadamente 30 épocas de treinamento.",
    "A análise post-hoc revelou que os resultados são robustos a diferentes sementes aleatórias.",
    "O poder estatístico do estudo foi calculado como 0,95 para detectar diferenças de 10%.",
    "O tamanho do efeito (Cohen's d) foi calculado como 0,85, indicando efeito grande.",
    "Os resultados foram validados utilizando o método de Bonferroni para correção de múltiplas comparações.",
    "A análise de subgrupos revelou que os resultados são consistentes entre diferentes faixas etárias."
]

# ==========================================
# ESTILO NARRATIVO (50 exemplos)
# ==========================================

textos_narrativo = [
    "Analisamos os dados coletados durante o experimento e percebemos padrões interessantes.",
    "Exploramos diferentes configurações do modelo e encontramos resultados promissores.",
    "Investigamos a influência do contexto e observamos que ele é fundamental para o desempenho.",
    "Avaliamos nossa abordagem em múltiplos corpora e verificamos sua eficácia.",
    "Implementamos um novo algoritmo que, em nossos testes, superou as alternativas existentes.",
    "Comparamos nossa metodologia com técnicas state-of-the-art e obtivemos resultados superiores.",
    "Testamos nossa hipótese em diferentes cenários e confirmamos nossas expectativas iniciais.",
    "Validamos nossa abordagem com especialistas da área e recebemos feedback positivo.",
    "Aplicamos o modelo proposto em problemas reais e obtivemos resultados encorajadores.",
    "Desenvolvemos uma solução que atende às necessidades identificadas em nossa pesquisa.",
    "Acreditamos que nossos resultados abrem novas perspectivas para pesquisas futuras.",
    "Consideramos que a abordagem desenvolvida representa um avanço significativo na área.",
    "Pensamos que as limitações identificadas não comprometem a validade das conclusões.",
    "Entendemos que ainda há espaço para melhorias, especialmente no pré-processamento.",
    "Refletimos sobre as implicações éticas do uso de modelos de linguagem em larga escala.",
    "Acreditamos que nossa contribuição pode beneficiar outros pesquisadores da comunidade.",
    "Consideramos importante compartilhar nosso código e dados para promover reprodutibilidade.",
    "Pensamos que a interpretabilidade dos modelos é um desafio crucial a ser enfrentado.",
    "Acreditamos que trabalhos futuros devem investigar a aplicação em outros domínios.",
    "Refletimos sobre como nossa abordagem se alinha com teorias linguísticas estabelecidas.",
    "Percebemos que o desempenho do modelo varia significativamente com o tamanho do corpus.",
    "Observamos que a remoção de stopwords teve impacto modesto nos resultados finais.",
    "Notamos que modelos pré-treinados em português superam aqueles treinados em multilíngue.",
    "Verificamos que o fine-tuning com poucos exemplos já produz resultados razoáveis.",
    "Constamos que a normalização dos dados é crucial para a estabilidade do treinamento.",
    "Detectamos que certos tipos de erro são sistemáticos e merecem investigação adicional.",
    "Identificamos padrões que sugerem a necessidade de uma abordagem híbrida.",
    "Descobrimos que o contexto local é mais importante que o contexto global para esta tarefa.",
    "Confirmamos nossa hipótese de que a arquitetura proposta é mais eficiente.",
    "Validamos empiricamente as vantagens teóricas esperadas do nosso método.",
    "Propomos uma nova arquitetura que combina o melhor de diferentes abordagens.",
    "Sugerimos que pesquisas futuras investiguem a aplicação em dados multimodais.",
    "Recomendamos a adoção de nossas diretrizes para anotação de corpus.",
    "Defendemos que a comunidade adote práticas mais rigorosas de avaliação.",
    "Apresentamos uma análise detalhada que esperamos ser útil para outros pesquisadores.",
    "Compartilhamos nossas implementações para facilitar a reprodução dos experimentos.",
    "Disponibilizamos nossos dados anotados para promover avanços na área.",
    "Convidamos a comunidade a explorar as muitas questões em aberto identificadas.",
    "Encaminhamos nossa pesquisa para aplicações práticas em sistemas reais.",
    "Visualizamos um futuro onde modelos como este serão ubíquos em aplicações de linguagem.",
    "Começamos nossa pesquisa com uma revisão sistemática da literatura especializada.",
    "Selecionamos cuidadosamente os conjuntos de dados que melhor representam o domínio.",
    "Projetamos experimentos para testar cada uma de nossas hipóteses de pesquisa.",
    "Coletamos dados de múltiplas fontes para garantir diversidade e representatividade.",
    "Processamos os dados utilizando pipelines que desenvolvemos especificamente para este fim.",
    "Treinamos nossos modelos em infraestrutura de GPU de última geração.",
    "Avaliamos os resultados utilizando métricas estabelecidas pela comunidade.",
    "Interpretamos os achados à luz das teorias existentes e de nossas contribuições.",
    "Documentamos todo o processo para garantir transparência e reprodutibilidade.",
    "Divulgamos nossos resultados em conferências e periódicos de alto impacto."
]

# ==========================================
# ESTILO DESCRITIVO (50 exemplos)
# ==========================================

textos_descritivo = [
    "O corpus contém 10.000 documentos. Cada documento possui 512 tokens.",
    "A acurácia foi de 94,5%. O desvio padrão é 0,03. O intervalo de confiança é 95%.",
    "Precisão: 97,3%. Recall: 94,1%. F1: 95,7%. AUC: 0,96.",
    "Média: 85,4. Mediana: 87,2. Variância: 12,5. Desvio: 3,54.",
    "Experimento A: n=1000, média=75,2. Experimento B: n=1000, média=78,4.",
    "Tempo de treinamento: 2h30min. Número de parâmetros: 110M. Memória: 12GB.",
    "Batch size: 32. Learning rate: 2e-5. Épocas: 10. Dropout: 0,1.",
    "CPU: Intel i7-10700. GPU: NVIDIA RTX 3080. RAM: 32GB. Tempo: 45min.",
    "Erro quadrático médio: 0,023. Erro absoluto médio: 0,112. R²: 0,94.",
    "Sensibilidade: 0,89. Especificidade: 0,92. Valor preditivo positivo: 0,91.",
    "Etapa 1: pré-processamento. Etapa 2: tokenização. Etapa 3: classificação.",
    "Primeiro, carregar dados. Segundo, normalizar. Terceiro, treinar. Quarto, testar.",
    "Passo 1: coletar corpus. Passo 2: anotar dados. Passo 3: treinar modelo.",
    "Fase 1: preparação. Fase 2: experimentação. Fase 3: análise. Fase 4: documentação.",
    "1º extrair features. 2º normalizar. 3º aplicar PCA. 4º classificar. 5º avaliar.",
    "Inicialmente, carregar bibliotecas. Em seguida, preparar dados. Depois, treinar modelo.",
    "Primeira etapa: coleta. Segunda etapa: limpeza. Terceira etapa: modelagem.",
    "Nível 1: básico. Nível 2: intermediário. Nível 3: avançado.",
    "Dia 1: planejamento. Dia 2: implementação. Dia 3: testes. Dia 4: documentação.",
    "Sprint 1: análise. Sprint 2: desenvolvimento. Sprint 3: validação.",
    "Resultados: 85% de acurácia. 78% de precisão. 82% de recall.",
    "Classificação: classe A (45%), classe B (32%), classe C (23%).",
    "Distribuição: treino (60%), validação (20%), teste (20%).",
    "Estatísticas: mínimo=12, máximo=98, média=54,3, mediana=51,0.",
    "Correlações: X1 (0,45), X2 (0,67), X3 (0,23), X4 (0,89).",
    "Tabela 1: resultados por configuração. Tabela 2: análise de erro.",
    "Figura 1: curva ROC (AUC=0,94). Figura 2: matriz de confusão.",
    "Comparação: método A (92%), método B (88%), método C (85%).",
    "Aumento de 15% em relação ao baseline. Redução de 23% no erro.",
    "Tempo de inferência: 0,23ms por exemplo. Throughput: 4340 exemplos/segundo.",
    "Execute: python train.py --config config.yaml --epochs 50.",
    "Comando: pip install -r requirements.txt. Depois: python main.py.",
    "Instalação: conda create -n env python=3.8. conda activate env.",
    "Configuração: edite o arquivo .env com suas credenciais.",
    "Uso: from modelo import Classificador; c = Classificador()",
    "Parâmetros: --input data/ --output results/ --model bert-base",
    "Logs: verificar arquivo logs/experiment_20241215.log",
    "Erro: ValueError: incompatible shapes. Solução: verificar dimensões.",
    "Aviso: deprecation warning. Atualizar para versão 2.0.",
    "Nota: resultados podem variar com diferentes sementes aleatórias.",
    "3 experimentos. 5 repetições. 2 condições. Total: 30 medições.",
    "Tamanho: pequeno (<1K), médio (1K-10K), grande (>10K).",
    "Temperatura: 25°C. Umidade: 45%. Pressão: 1013 hPa.",
    "Duração: curta (<1h), média (1-4h), longa (>4h).",
    "Custo: baixo (<$100), médio ($100-$1000), alto (>$1000).",
    "Prioridade: alta (1), média (2), baixa (3). Status: pendente (0), concluído (1).",
    "Versão: 1.0.0 (estável), 2.0.0-beta (experimental).",
    "Licença: MIT (código aberto), CC-BY-4.0 (dados).",
    "Formato: JSON (dados), YAML (configuração), Markdown (documentação).",
    "Encoding: UTF-8. Line ending: LF. Indentação: 2 espaços."
]

print(f"Acadêmico: {len(textos_academico)} exemplos")
print(f"Narrativo: {len(textos_narrativo)} exemplos")
print(f"Descritivo: {len(textos_descritivo)} exemplos")

# Combinar todos os textos
todos_textos = textos_academico + textos_narrativo + textos_descritivo
todos_labels = (["academico"] * len(textos_academico) +
                ["narrativo"] * len(textos_narrativo) +
                ["descritivo"] * len(textos_descritivo))

print(f"\nTotal de exemplos para fine-tuning: {len(todos_textos)}")

# ============================================
# 3. ANALISAR CORPUS (PALAVRAS MAIS FREQUENTES)
# ============================================

print("\n" + "="*60)
print("3. ANALISANDO CORPUS")
print("="*60)

todos_tokens = []
substantivos = []
verbos = []

for artigo in articles:
    tokens = artigo.get("artigo_tokenizado", [])
    pos_tags = artigo.get("pos_tagger", [])
    for token, pos in zip(tokens, pos_tags):
        if len(token) > 2:
            todos_tokens.append(token.lower())
            if pos == 'NOUN':
                substantivos.append(token.lower())
            elif pos == 'VERB':
                verbos.append(token.lower())

freq_tokens = Counter(todos_tokens)
freq_subst = Counter(substantivos)
freq_verb = Counter(verbos)

substantivo_top = freq_subst.most_common(1)[0][0] if substantivos else "dados"
verbo_top = freq_verb.most_common(1)[0][0] if verbos else "pode"

print(f"Total de tokens únicos: {len(freq_tokens)}")
print(f"Substantivo mais frequente: {substantivo_top}")
print(f"Verbo mais frequente: {verbo_top}")

palavras_atv1 = ["modelos", "linguagem", substantivo_top, verbo_top]
print(f"\nPalavras para análise: {palavras_atv1}")

# ============================================
# 4. FUNÇÕES PARA OLLAMA
# ============================================

def obter_embedding(texto, modelo=MODELO_EMBEDDING, max_retries=3):
    """Obtém embedding usando Ollama com retry"""
    for attempt in range(max_retries):
        try:
            response = requests.post(
                f"{OLLAMA_URL}/api/embeddings",
                json={"model": modelo, "prompt": texto},
                timeout=60
            )
            if response.status_code == 200:
                return np.array(response.json()["embedding"])
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                print(f"Erro no embedding: {e}")
    return np.random.randn(1024)

def qwen_generate(prompt, temperature=0.1, modelo=MODELO_USAR, max_retries=3):
    """Gera texto usando Qwen via Ollama com retry"""
    for attempt in range(max_retries):
        try:
            response = requests.post(
                f"{OLLAMA_URL}/api/generate",
                json={
                    "model": modelo,  # Agora usa MODELO_USAR (que pode ser o fine-tunado)
                    "prompt": prompt,
                    "temperature": temperature,
                    "stream": False
                },
                timeout=120
            )
            if response.status_code == 200:
                return response.json()["response"].strip()
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Tentando novamente ({attempt+1}/{max_retries})...")
                time.sleep(3)
            else:
                print(f"  Erro: {e}")
    return ""

# ============================================
# 5. ATIVIDADE 1: VETORES GERADOS
# ============================================

print("\n" + "="*60)
print("ATIVIDADE 1: VETORES GERADOS")
print("="*60)

vetores_qwen = {}

for p in palavras_atv1:
    vetor = obter_embedding(p)
    vetores_qwen[p] = vetor
    print(f"\n{p}:")
    print(f"  Dimensão: {len(vetor)}")
    print(f"  Primeiras 10 dim: {np.round(vetor[:10], 4)}")
    print(f"  Norma: {np.linalg.norm(vetor):.4f}")

# ============================================
# 6. ATIVIDADE 2: TERMOS MAIS SIMILARES
# ============================================

print("\n" + "="*60)
print("ATIVIDADE 2: TERMOS MAIS SIMILARES")
print("="*60)

def calcular_similaridades(vetor_alvo, tokens_freq, top_n=5):
    """Calcula similaridade usando embeddings"""
    tokens_validos = [t for t in list(tokens_freq.keys())
                     if len(t) > 2 and t not in palavras_atv1]
    tokens_validos = tokens_validos[:200]

    resultados = []
    for token in tokens_validos:
        try:
            vetor_token = obter_embedding(token)
            sim = cosine_similarity([vetor_alvo], [vetor_token])[0][0]
            resultados.append((token, sim))
        except:
            continue

    resultados.sort(key=lambda x: x[1], reverse=True)
    return resultados[:top_n]

resultados_similaridades = {}

for p in palavras_atv1:
    print(f"\n  Similares a '{p}':")
    similares = calcular_similaridades(vetores_qwen[p], freq_tokens, top_n=5)
    resultados_similaridades[p] = similares
    for i, (token, score) in enumerate(similares, 1):
        print(f"    {i}. {token} ({score:.4f})")

# ============================================
# 7. CLASSIFICAR ARTIGOS COM MODELO FINETUNADO
# ============================================

print("\n" + "="*60)
print("ATIVIDADE 3: CLASSIFICANDO ARTIGOS")
print("="*60)

def classificar_artigo(texto, modelo=MODELO_USAR):
    """Classifica um artigo usando o modelo disponível"""

    # Se estamos usando o modelo base (sem fine-tune), usar palavras-chave
    if modelo == MODELO_BASE:
        print("  Usando classificação por palavras-chave (fallback)")
        return classificar_por_palavras_chave(texto)

    exemplos_estilos = """
    ESTILO ACADEMICO: Uso frequente da terceira pessoa do singular ou da voz passiva. Evita adjetivos, jargoes, girias e superlativos desnecessarios.

    Exemplos Academicos:

    1. "Observou-se que os resultados apresentam significancia estatistica."
    2. "Foi verificado que os modelos baseados em transformer superam abordagens anteriores."
    3. "Conclui-se que a metodologia empregada demonstra eficacia na tarefa proposta."
    4. "Os dados foram coletados e analisados estatisticamente segundo protocolos estabelecidos."
    5. "Realizou-se uma analise detalhada dos componentes principais do modelo."
    6. "Verificou-se uma correlacao significativa entre as variaveis analisadas."
    7. "Pode-se observar que os resultados obtidos sao consistentes com a literatura."
    8. "Nota-se uma tendencia clara de melhoria no desempenho dos classificadores."
    9. "Foi demonstrado que a abordagem proposta e eficaz para o problema em questao."
    10. "Conduziu-se um experimento controlado para avaliar o impacto das configuracoes."
    11. "E importante ressaltar que os resultados devem ser interpretados com cautela."
    12. "Considera-se que a amostra utilizada e representativa da populacao estudada."
    13. "Entende-se que os achados contribuem significativamente para a area de conhecimento."
    14. "Salienta-se a necessidade de replicacao dos experimentos em diferentes contextos."
    15. "Destaca-se a relevancia dos resultados obtidos para aplicacoes praticas."
    16. "Ressalta-se que as limitacoes do estudo nao comprometem as conclusoes principais."
    17. "Argumenta-se que os modelos neurais apresentam vantagens sobre metodos tradicionais."
    18. "Sugere-se que pesquisas futuras investiguem a generalizacao dos resultados."
    19. "Infere-se dos dados que existe uma relacao causal entre as variaveis estudadas."
    20. "Depreende-se da analise que os resultados sao robustos a diferentes configuracoes."

    ESTILO NARRATIVO: Escrita fluida, reflexiva, uso da primeira pessoa do plural. Aproxima-se de um ensaio academico.

    Exemplos Narrativos:

    1. "Analisamos os dados coletados durante o experimento e percebemos padroes interessantes."
    2. "Exploramos diferentes configuracoes do modelo e encontramos resultados promissores."
    3. "Investigamos a influencia do contexto e observamos que ele e fundamental para o desempenho."
    4. "Avaliamos nossa abordagem em multiplos corpora e verificamos sua eficacia."
    5. "Implementamos um novo algoritmo que, em nossos testes, superou as alternativas existentes."
    6. "Comparamos nossa metodologia com tecnicas state-of-the-art e obtivemos resultados superiores."
    7. "Testamos nossa hipotese em diferentes cenarios e confirmamos nossas expectativas iniciais."
    8. "Validamos nossa abordagem com especialistas da area e recebemos feedback positivo."
    9. "Aplicamos o modelo proposto em problemas reais e obtivemos resultados encorajadores."
    10. "Desenvolvemos uma solucao que atende as necessidades identificadas em nossa pesquisa."
    11. "Acreditamos que nossos resultados abrem novas perspectivas para pesquisas futuras."
    12. "Consideramos que a abordagem desenvolvida representa um avanco significativo na area."
    13. "Pensamos que as limitacoes identificadas nao comprometem a validade das conclusoes."
    14. "Entendemos que ainda ha espaco para melhorias, especialmente no pre-processamento."
    15. "Refletimos sobre as implicacoes eticas do uso de modelos de linguagem em larga escala."
    16. "Acreditamos que nossa contribuicao pode beneficiar outros pesquisadores da comunidade."
    17. "Consideramos importante compartilhar nosso codigo e dados para promover reprodutibilidade."
    18. "Pensamos que a interpretabilidade dos modelos e um desafio crucial a ser enfrentado."
    19. "Acreditamos que trabalhos futuros devem investigar a aplicacao em outros dominios."
    20. "Refletimos sobre como nossa abordagem se alinha com teorias linguisticas estabelecidas."

    ESTILO DESCRITIVO: Rigor extremo, frases curtas, objetividade matematica ou estatistica e ordenacao cronologica/logica.

    Exemplos Descritivos:

    1. "O corpus contem 10.000 documentos. Cada documento possui 512 tokens."
    2. "A acuracia foi de 94,5 por cento. O desvio padrao e 0,03. O intervalo de confianca e 95 por cento."
    3. "Precisao: 97,3 por cento. Recall: 94,1 por cento. F1: 95,7 por cento. AUC: 0,96."
    4. "Media: 85,4. Mediana: 87,2. Variancia: 12,5. Desvio: 3,54."
    5. "Experimento A: n=1000, media=75,2. Experimento B: n=1000, media=78,4."
    6. "Tempo de treinamento: 2h30min. Numero de parametros: 110M. Memoria: 12GB."
    7. "Batch size: 32. Learning rate: 2e-5. Epocas: 10. Dropout: 0,1."
    8. "CPU: Intel i7-10700. GPU: NVIDIA RTX 3080. RAM: 32GB. Tempo: 45min."
    9. "Erro quadratico medio: 0,023. Erro absoluto medio: 0,112. R ao quadrado: 0,94."
    10. "Sensibilidade: 0,89. Especificidade: 0,92. Valor preditivo positivo: 0,91."
    11. "Etapa 1: pre-processamento. Etapa 2: tokenizacao. Etapa 3: classificacao."
    12. "Primeiro, carregar dados. Segundo, normalizar. Terceiro, treinar. Quarto, testar."
    13. "Passo 1: coletar corpus. Passo 2: anotar dados. Passo 3: treinar modelo."
    14. "Fase 1: preparacao. Fase 2: experimentacao. Fase 3: analise. Fase 4: documentacao."
    15. "Primeiro extrair features. Segundo normalizar. Terceiro aplicar PCA. Quarto classificar. Quinto avaliar."
    16. "Inicialmente, carregar bibliotecas. Em seguida, preparar dados. Depois, treinar modelo."
    17. "Primeira etapa: coleta. Segunda etapa: limpeza. Terceira etapa: modelagem."
    18. "Nivel 1: basico. Nivel 2: intermediario. Nivel 3: avancado."
    19. "Dia 1: planejamento. Dia 2: implementacao. Dia 3: testes. Dia 4: documentacao."
    20. "Sprint 1: analise. Sprint 2: desenvolvimento. Sprint 3: validacao."
    """


    prompt = f"""Classifique o texto abaixo em um dos tres estilos: ACADEMICO, NARRATIVO ou DESCRITIVO.

    {exemplos_estilos}

    Texto: {texto[:400]}

    Responda apenas com uma palavra: ACADEMICO, NARRATIVO ou DESCRITIVO."""

    resposta = qwen_generate(prompt, temperature=0.1, modelo=modelo)

    if resposta:
        resposta = resposta.strip().lower()
        if 'academico' in resposta:
            return 'academico'
        elif 'narrativo' in resposta:
            return 'narrativo'
        elif 'descritivo' in resposta:
            return 'descritivo'

    # Fallback por palavras-chave
    return classificar_por_palavras_chave(texto)

def classificar_por_palavras_chave(texto):
    print("Classificado por palavras-chave (fallback equilibrado)")
    texto_lower = texto.lower()

    palavras_academico = ['observou-se', 'foi verificado', 'conclui-se', 'realizou-se',
                          'verificou-se', 'nota-se', 'demonstrado', 'conduziu-se',
                          'salienta-se', 'destaca-se', 'ressalta-se']

    palavras_narrativo = ['analisamos', 'exploramos', 'investigamos', 'avaliamos',
                          'implementamos', 'comparamos', 'testamos', 'validamos',
                          'aplicamos', 'desenvolvemos', 'acreditamos', 'consideramos',
                          'pensamos', 'entendemos', 'refletimos']

    count_academico = sum(1 for p in palavras_academico if p in texto_lower)
    count_narrativo = sum(1 for p in palavras_narrativo if p in texto_lower)

    if count_academico > count_narrativo:
        return 'academico'
    elif count_narrativo > count_academico:
        return 'narrativo'
    else:
        return 'descritivo'

# Testar classificador
print("\nTestando classificador com modelo fine-tunado:")
testes = [
    ("Observou-se que os resultados são estatisticamente significativos (p<0,05).", "academico"),
    ("Analisamos os dados cuidadosamente e percebemos padrões muito interessantes.", "narrativo"),
    ("Acurácia: 94,5%. Precisão: 93,2%. Recall: 91,8%. F1: 92,3%.", "descritivo"),
    ("Foi demonstrado que o modelo proposto supera as abordagens baseline.", "academico"),
    ("Acreditamos que nossa contribuição pode beneficiar toda a comunidade científica.", "narrativo"),
    ("Passo 1: carregar. Passo 2: processar. Passo 3: analisar. Passo 4: concluir.", "descritivo")
]

for texto, esperado in testes:
    resultado = classificar_artigo(texto)
    print(f"\nTexto: {texto[:60]}...")
    print(f"  Esperado: {esperado}")
    print(f"  Classificado: {resultado}")

# ============================================
# 8. CLASSIFICAR TODOS OS ARTIGOS
# ============================================

print("\n" + "="*60)
print("CLASSIFICANDO TODOS OS ARTIGOS DO CORPUS")
print("="*60)

resultados_classificacao = []

for i, artigo in enumerate(articles, 1):
    titulo = artigo.get("titulo", f"Artigo {i}")
    texto = artigo.get("artigo_completo", "")

    if not texto or len(texto) < 200:
        continue

    estilo = classificar_artigo(texto[:500])

    resultados_classificacao.append({
        'id': i,
        'titulo': titulo,
        'estilo': estilo
    })

    if i % 10 == 0:
        print(f"  Processados {i} de {len(articles)} artigos...")

print(f"\n{len(resultados_classificacao)} artigos classificados!")

# ============================================
# 9. EXIBIR RESULTADOS
# ============================================

print("\n" + "="*60)
print("RESULTADOS DA CLASSIFICAÇÃO")
print("="*60)

print(f"\n{'#':3} | {'ESTILO':12} | TÍTULO")
print("-" * 70)

for r in resultados_classificacao[:20]:
    estilo_display = r['estilo'].upper()
    titulo_resumido = r['titulo'][:50] + "..." if len(r['titulo']) > 50 else r['titulo']
    print(f"{r['id']:3} | {estilo_display:12} | {titulo_resumido}")

# ============================================
# 10. DISTRIBUIÇÃO DOS ESTILOS
# ============================================

print("\n" + "="*60)
print("DISTRIBUIÇÃO DOS ESTILOS")
print("="*60)

contagem = Counter([r['estilo'] for r in resultados_classificacao])

print(f" ACADÊMICO:  {contagem.get('academico', 0)} artigos")
print(f" NARRATIVO:  {contagem.get('narrativo', 0)} artigos")
print(f" DESCRITIVO: {contagem.get('descritivo', 0)} artigos")
print(f"\n Total: {len(resultados_classificacao)} artigos analisados")

# ============================================
# 11. GRÁFICO DE DISTRIBUIÇÃO
# ============================================

print("\n" + "="*60)
print("GRÁFICO DE DISTRIBUIÇÃO DOS ESTILOS")
print("="*60)

df_plot = pd.DataFrame({
    'Estilo': ['Acadêmico', 'Narrativo', 'Descritivo'],
    'Quantidade': [
        contagem.get('academico', 0),
        contagem.get('narrativo', 0),
        contagem.get('descritivo', 0)
    ]
})

fig = px.bar(df_plot, x='Estilo', y='Quantidade',
             title='Distribuição de Estilos de Escrita (Qwen 3.5 Fine-tuned)',
             color='Estilo',
             color_discrete_map={'Acadêmico': '#2E86AB', 'Narrativo': '#A23B72', 'Descritivo': '#F18F01'},
             text='Quantidade')
fig.update_layout(width=600, height=450)
fig.show()

# ============================================
# 12. MATRIZ DE CONFUSÃO
# ============================================

print("\n" + "="*60)
print("MATRIZ DE CONFUSÃO DA CLASSIFICAÇÃO")
print("="*60)

label_map = {"academico": 0, "narrativo": 1, "descritivo": 2}
label_names = ["Acadêmico", "Narrativo", "Descritivo"]

# Labels reais baseados em palavras-chave
labels_reais = []
for artigo in articles[:len(resultados_classificacao)]:
    texto = artigo.get("artigo_completo", "").lower()
    if "observou-se" in texto or "foi verificado" in texto or "conclui-se" in texto:
        labels_reais.append("academico")
    elif "analisamos" in texto or "acreditamos" in texto or "percebemos" in texto:
        labels_reais.append("narrativo")
    else:
        labels_reais.append("descritivo")

labels_preditos = [r['estilo'] for r in resultados_classificacao]

y_true = [label_map[l] for l in labels_reais]
y_pred = [label_map[l] for l in labels_preditos]

cm = confusion_matrix(y_true, y_pred)

print("\nMatriz de Confusão (valores absolutos):")
print("                  Predito")
print("                  Acadêmico  Narrativo  Descritivo")
print(f"Real  Acadêmico   {cm[0,0]:>9}  {cm[0,1]:>9}  {cm[0,2]:>9}")
print(f"      Narrativo   {cm[1,0]:>9}  {cm[1,1]:>9}  {cm[1,2]:>9}")
print(f"      Descritivo  {cm[2,0]:>9}  {cm[2,1]:>9}  {cm[2,2]:>9}")

print("\n Métricas por Classe:")
print("-" * 40)

for i, nome in enumerate(label_names):
    tp = cm[i, i]
    fp = sum(cm[:, i]) - tp
    fn = sum(cm[i, :]) - tp

    precisao = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precisao * recall) / (precisao + recall) if (precisao + recall) > 0 else 0

    print(f"\n  {nome}:")
    print(f"    Precisão: {precisao:.2%}")
    print(f"    Recall:   {recall:.2%}")
    print(f"    F1-Score: {f1:.2%}")

acuracia = np.trace(cm) / np.sum(cm)
print(f"\n Acurácia Geral: {acuracia:.2%}")

# ============================================
# 13. GRÁFICO DA MATRIZ DE CONFUSÃO
# ============================================

print("\n Gráfico da Matriz de Confusão")

fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=label_names,
    y=label_names,
    annotation_text=cm,
    colorscale='Blues',
    showscale=True,
    font_colors=['white', 'black']
)

fig_cm.update_layout(
    title=dict(
        text="Matriz de Confusão - Classificação de Estilos (Qwen 3.5 Fine-tuned)",
        font=dict(size=18)
    ),
    width=600,
    height=500,
    xaxis=dict(title="Predito", side="bottom"),
    yaxis=dict(title="Real", autorange="reversed")
)

fig_cm.show()

# ============================================
# 14. RELATÓRIO DE CLASSIFICAÇÃO
# ============================================

print("\n Relatório de Classificação")
print("-" * 40)
report = classification_report(y_true, y_pred, target_names=label_names, digits=4)
print(report)

# ============================================
# 15. RESUMO DAS RESPOSTAS
# ============================================

print("\n" + "="*60)
print("RESUMO DAS RESPOSTAS - AP6 (QWEN 3.5 FINETUNADO)")
print("="*60)

print("\n1) VETORES GERADOS:")
for p in palavras_atv1:
    print(f"   {p}: vetor {len(vetores_qwen[p])}D")

print("\n2) TERMOS MAIS SIMILARES:")
for p, sims in resultados_similaridades.items():
    print(f"   {p}: {', '.join([s[0] for s in sims[:5]])}")

print(f"""
3) CLASSIFICAÇÃO DE ESTILOS:
   Tarefa: Classificação ternária de textos
   Modelo: Qwen 3.5 via Ollama (Fine-tuned)
   Modelo usado: {MODELO_USAR}
   Dataset de treino: {len(todos_textos)} exemplos (50 por estilo)
   Artigos classificados: {len(resultados_classificacao)}
   Distribuição: Acadêmico={contagem.get('academico', 0)}, Narrativo={contagem.get('narrativo', 0)}, Descritivo={contagem.get('descritivo', 0)}
   Acurácia: {acuracia:.2%}
""")